# 01.05_processing_2021Clytia

整理 Clytia 数据与细胞元数据。

- 当前文件：`analysis/01_preprocessing/01.05_processing_2021Clytia.ipynb`
- 原始来源：`Codes/01.05_processing_2021Clytia.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`pandas`, `scanpy`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


# Clytia

## 1.读取原始数据

In [ ]:
import scanpy as sc
import pandas as pd

# 正确读取并转置表达矩阵
adata = sc.read_csv('/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2021Clytia/filter_matrix/exprMatrix.tsv', delimiter='\t', first_column_names=True)
adata

In [ ]:
# 转置为标准的h5ad的adata
adata = adata.T
adata

In [ ]:
adata.var

In [ ]:
# 查找重复的基因名
duplicates = adata.var_names[adata.var_names.duplicated()]
# 输出重复的基因名
print("重复的基因名：", duplicates)
# 删除重复的基因（保留第一个出现的）
adata = adata[:, ~adata.var_names.duplicated()]
adata.var

In [ ]:
# 基因重命名，更换_为-
adata.var_names = adata.var_names.str.replace('_', '-', regex=False)
adata.var

In [ ]:
# 细胞名
adata.obs

In [ ]:
# 矩阵内容
# Values were log1p-normalized, mean-centered and scaled, and filtered for highly variable genes using the same procedure described above for downstream dimensionality reduction and visualization (e.g., PCA) using Scanpy. 
adata.X

## 2.添加细胞信息

In [ ]:
# 读取元数据
meta_data = pd.read_csv('/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2021Clytia/filter_matrix/meta.tsv', delimiter='\t', index_col='cellId')
meta_data.head()

In [ ]:
# 确保元数据和AnnData的列名对应
print("AnnData cells:", adata.obs_names.shape)
print("Metadata entries:", meta_data.shape)

In [ ]:
adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True)
adata

In [ ]:
# 读取UMAP坐标并赋值
umap_coords = pd.read_csv('/share/home/zhangze/zz/NeuralOrigin/Data/01.RawData/PublicData/2021Clytia/filter_matrix/UMAP.coords.tsv', delimiter='\t', header=None, index_col=0)
adata.obsm['X_umap'] = umap_coords.loc[adata.obs_names].values  # 确保索引对齐
adata

In [ ]:
sc.pl.umap(adata, color=['batch', 'orgID', 'UMI Count'])

In [ ]:
sc.pl.umap(adata, color=['cellRanger_louvain', 'annos'])

In [ ]:
sc.pl.umap(adata, color='annosSub')

In [ ]:
sc.pl.umap(adata, color='annos')

In [ ]:
adata.obs

In [ ]:
print(len(adata.obs["cellRanger_louvain"].value_counts()))
adata.obs["cellRanger_louvain"].value_counts()

In [ ]:
print(len(adata.obs["orgID"].value_counts()))
adata.obs["orgID"].value_counts()

In [ ]:
print(len(adata.obs["annos"].value_counts()))
adata.obs["annos"].value_counts()

In [ ]:
print(len(adata.obs["annosSub"].value_counts()))
adata.obs["annosSub"].value_counts()

In [ ]:
adata

In [ ]:
# 保存adata数据
raw_clytia_path = "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Clhe.normalized.h5ad"
adata.write(raw_clytia_path)

In [ ]:
# 导出基因id
adata.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/SingleCellDataH5ad/Clhe.genes.txt', index=False, header=False)